In [5]:
!pip install -q transformers datasets accelerate scikit-learn pandas torch

import json
import torch
import pandas as pd

from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit,train_test_split
from sklearn.metrics import accuracy_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments, Trainer
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

GPU available: False
Running on CPU


In [7]:
DATASET_PATH = "/content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/data/raw/ethan_tickets_10000.csv"

df = pd.read_csv(DATASET_PATH)

print("Total tickets:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Total tickets: 10000
Columns: ['ticket_id', 'text', 'resolver_group', 'priority', 'parent_incident_id', 'client_id', 'caller_name']


,ticket_id,text,resolver_group,priority,parent_incident_id,client_id,caller_name
0,TKT006253,The server reports a hardware failure involvin...,Hardware Support,P3,INC03557,Client_B,User_4770
1,TKT004685,The web server fails to serve pages to externa...,Web Hosting,P3,INC02656,Client_B,User_4558
2,TKT001732,Users receive an installation error when updat...,Software Support,P2,INC00970,Client_B,User_8927
3,TKT004743,The collaboration platform fails to load share...,Email and Collaboration,P3,INC02693,Client_C,User_1042
4,TKT004522,The security system reports an unexpected chan...,IT Security Operations,P3,INC02560,Client_A,User_7891


In [8]:

resolver_labels = sorted(df["resolver_group"].unique())
priority_labels = sorted(df["priority"].unique())

resolver_mapping = {label: index for index, label in enumerate(resolver_labels)}
priority_mapping = {label: index for index, label in enumerate(priority_labels)}

df["resolver_label"] = df["resolver_group"].map(resolver_mapping)
df["priority_label"] = df["priority"].map(priority_mapping)

print("Resolver classes:", resolver_labels)
print("Priority classes:", priority_labels)

print("Resolver class count:", len(resolver_labels))
print("Priority class count:", len(priority_labels))

train, test = train_test_split(df, test_size=0.2,shuffle=True)


print("Training records:", len(train))
print("Testing records:", len(test))


Resolver classes: ['Application Support', 'Backup and Recovery', 'CRM Support', 'Cloud Infrastructure', 'Cybersecurity', 'Data Engineering', 'Database Administration', 'Desktop Support', 'DevOps', 'ERP Support', 'Email and Collaboration', 'Hardware Support', 'IT Security Operations', 'IT Service Desk', 'Identity and Access Management', 'Linux Administration', 'Monitoring and Observability', 'Network Operations', 'Software Support', 'Storage Administration', 'Telecommunications', 'Virtualization Support', 'Web Hosting', 'Windows Administration']
Priority classes: ['P1', 'P2', 'P3', 'P4']
Resolver class count: 24
Priority class count: 4
Training records: 8000
Testing records: 2000


In [9]:
MODEL_NAME = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Tokenizer loaded successfully


In [10]:
def enc(data):

    dataset = Dataset.from_dict({
        "text": data["text"].tolist(),
        "resolver_labels": data["resolver_label"].tolist(),
        "priority_labels": data["priority_label"].tolist()
    })

    dataset = dataset.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=128
        ),
        batched=True
    )

    return dataset.remove_columns(["text"])


train_dataset = enc(train)
test_dataset = enc(test)

print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training dataset: 8000
Testing dataset: 2000


In [11]:

import torch
from torch import nn
from transformers import BertModel, BertPreTrainedModel


class EthanMultiTaskBERT(BertPreTrainedModel):

    def __init__(self, config):
        super().__init__(config)

        # Shared BERT encoder
        self.bert = BertModel(config)

        # Number of output classes
        self.num_resolver_labels = config.num_resolver_labels
        self.num_priority_labels = config.num_priority_labels

        # Resolver classification head
        self.resolver_classifier = nn.Linear(
            config.hidden_size,
            self.num_resolver_labels
        )

        # Priority classification head
        self.priority_classifier = nn.Linear(
            config.hidden_size,
            self.num_priority_labels
        )

        # Loss function
        self.loss_fn = nn.CrossEntropyLoss()

        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        resolver_labels=None,
        priority_labels=None
    ):

        # Extract shared features
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        pooled_output = outputs.pooler_output

        # Make both predictions
        resolver_logits = self.resolver_classifier(pooled_output)
        priority_logits = self.priority_classifier(pooled_output)

        loss = None

        # Calculate combined training loss
        if resolver_labels is not None and priority_labels is not None:

            resolver_loss = self.loss_fn(
                resolver_logits,
                resolver_labels
            )

            priority_loss = self.loss_fn(
                priority_logits,
                priority_labels
            )

            loss = resolver_loss + priority_loss

        if loss is not None:
            return loss, resolver_logits, priority_logits

        return resolver_logits, priority_logits


print("EthanMultiTaskBERT class defined successfully!")


EthanMultiTaskBERT class defined successfully!


In [12]:
from transformers import BertConfig

# Load BERT configuration
config = BertConfig.from_pretrained(MODEL_NAME)

# Number of classes for each prediction task
config.num_resolver_labels = len(resolver_labels)
config.num_priority_labels = len(priority_labels)

# Resolver label mappings
config.resolver_id2label = {
    i: label for i, label in enumerate(resolver_labels)
}

config.resolver_label2id = {
    label: i for i, label in enumerate(resolver_labels)
}

# Priority label mappings
config.priority_id2label = {
    i: label for i, label in enumerate(priority_labels)
}

config.priority_label2id = {
    label: i for i, label in enumerate(priority_labels)
}

# Standard Hugging Face mappings (resolver task)
config.id2label = config.resolver_id2label
config.label2id = config.resolver_label2id

# Initialize the multi-task BERT model
model = EthanMultiTaskBERT.from_pretrained(
    MODEL_NAME,
    config=config
)

print("Multi-task BERT loaded successfully!")
print("Resolver classes:", config.num_resolver_labels)
print("Priority classes:", config.num_priority_labels)

[transformers] You are using a model of type `modernbert` to instantiate a model of type `bert`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights: 0it [00:00, ?it/s]

[transformers] EthanMultiTaskBERT LOAD REPORT from: answerdotai/ModernBERT-base
Key                                                           | Status     | 
--------------------------------------------------------------+------------+-
model.layers.{1...21}.attn_norm.weight                        | UNEXPECTED | 
model.layers.{0...21}.attn.Wqkv.weight                        | UNEXPECTED | 
model.layers.{0...21}.mlp.Wo.weight                           | UNEXPECTED | 
model.layers.{0...21}.mlp_norm.weight                         | UNEXPECTED | 
model.layers.{0...21}.mlp.Wi.weight                           | UNEXPECTED | 
model.layers.{0...21}.attn.Wo.weight                          | UNEXPECTED | 
model.embeddings.tok_embeddings.weight                        | UNEXPECTED | 
head.dense.weight                                             | UNEXPECTED | 
decoder.bias                                                  | UNEXPECTED | 
model.embeddings.norm.weight                                  

Multi-task BERT loaded successfully!
Resolver classes: 24
Priority classes: 4


In [13]:
from sklearn.metrics import accuracy_score

def compute_metrics(prediction):

    # Extract predictions from both classification heads
    resolver_logits, priority_logits = prediction.predictions

    # Extract actual labels
    resolver_labels, priority_labels = prediction.label_ids

    # Convert logits into predicted classes
    resolver_predictions = resolver_logits.argmax(axis=-1)
    priority_predictions = priority_logits.argmax(axis=-1)

    return {
        "resolver_accuracy": accuracy_score(
            resolver_labels,
            resolver_predictions
        ),

        "priority_accuracy": accuracy_score(
            priority_labels,
            priority_predictions
        )
    }

In [14]:
training_args = TrainingArguments(
    output_dir="/content/out",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

print("Training parameters configured")

Training parameters configured


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer initialized successfully")

Trainer initialized successfully


In [ ]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
results = trainer.evaluate()

print("\nEvaluation Results")

for key, value in results.items():
    print(f"{key}: {value}")

Training Loss,Validation Loss,Epoch,Resolver Accuracy,Priority Accuracy
0.037388,0.024741,3,1.000000,1.000000


KeyError: 'eval_accuracy'

In [ ]:
import os
import json

MODEL_OUTPUT = "/content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/models/ethan_resolver_model_modern"
# MODEL_OUTPUT = "/content/EthanGlobal_AI_Triage_Agent/models#"

os.makedirs(MODEL_OUTPUT, exist_ok=True)

trainer.save_model(MODEL_OUTPUT)
tokenizer.save_pretrained(MODEL_OUTPUT)

#with open(f"{MODEL_OUTPUT}/labels.json", "w") as file:
    #json.dump(labels, file)

print("Model saved successfully!")
print("Location:", MODEL_OUTPUT)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Location: /content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/models/ethan_resolver_model
